# Chapter 14 Companion Notebook: Natural Language Processing in Finance

This notebook reproduces every worked example in Chapter 14 from scratch, in the order they appear in the chapter text, including the toy-embedding vector-arithmetic analogy, a plain-English disclosure rewrite comparison, causal-vs-bidirectional attention (encoder-only vs. decoder-only architectures), RAG evaluation metrics (precision@k and a faithfulness score), LoRA parameter-efficient fine-tuning, a three-step agent that cross-checks its own computed P/E ratio, and an API-versus-self-hosting deployment cost comparison.

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. Bag-of-words and TF-IDF

In [1]:
import numpy as np
import pandas as pd

docs = {
    "D1": "revenue growth exceeded expectations this quarter",
    "D2": "management is cautious about revenue growth given the headwinds",
    "D3": "management remains confident despite the headwinds this quarter",
    "D4": "expectations were not met and revenue declined",
}
vocab = ["revenue", "growth", "headwinds", "expectations", "quarter", "management"]

tf = pd.DataFrame(0, index=docs.keys(), columns=vocab)
for d, text in docs.items():
    words = text.lower().replace(",", "").split()
    for w in vocab:
        tf.loc[d, w] = words.count(w)

tf

,revenue,growth,headwinds,expectations,quarter,management
D1,1,1,0,1,1,0
D2,1,1,1,0,0,1
D3,0,0,1,0,1,1
D4,1,0,0,1,0,0


In [2]:
N = len(docs)
df = (tf > 0).sum(axis=0)
idf = np.log(N / df)
tfidf = tf * idf
print("Document frequency:\n", df)
print("\nIDF = ln(N/df):\n", idf.round(4))
tfidf.round(4)

Document frequency:
 revenue         3
growth          2
headwinds       2
expectations    2
quarter         2
management      2
dtype: int64

IDF = ln(N/df):
 revenue         0.2877
growth          0.6931
headwinds       0.6931
expectations    0.6931
quarter         0.6931
management      0.6931
dtype: float64


,revenue,growth,headwinds,expectations,quarter,management
D1,0.2877,0.6931,0.0000,0.6931,0.6931,0.0000
D2,0.2877,0.6931,0.6931,0.0000,0.0000,0.6931
D3,0.0000,0.0000,0.6931,0.0000,0.6931,0.6931
D4,0.2877,0.0000,0.0000,0.6931,0.0000,0.0000


## 2. Financial sentiment scoring (Loughran-McDonald subset)

In [3]:
positive_words = {"strong", "growth", "exceeded", "confident", "improved"}
negative_words = {"cautious", "headwinds", "decline", "declined", "weak", "concern", "concerns"}

transcript_a = ("revenue growth was strong this quarter and results exceeded expectations "
                "across every segment we remain confident heading into next year")
transcript_b = ("management is cautious about headwinds and results declined modestly "
                "we have some concerns about demand next quarter")

for name, t in [("A (optimistic)", transcript_a), ("B (cautious)", transcript_b)]:
    words = t.split()
    pos = sum(1 for w in words if w in positive_words)
    neg = sum(1 for w in words if w in negative_words)
    total = len(words)
    tone = (pos - neg) / total
    print(f"Transcript {name}: words={total}, pos={pos}, neg={neg}, tone={tone:.4f}")

Transcript A (optimistic): words=20, pos=4, neg=0, tone=0.2000
Transcript B (cautious): words=17, pos=0, neg=4, tone=-0.2353


## 3. An event-driven sentiment trading signal

In [4]:
tone_scores = np.array([0.15, -0.10, 0.08, -0.05, 0.20, -0.18, 0.02, -0.12])
raw_returns = np.array([1.2, -0.8, 0.3, 0.4, 2.1, -1.5, -0.2, -0.9])  # percent, next-day return

signal = np.sign(tone_scores)
strategy_returns = signal * raw_returns

strat_mean, strat_std = strategy_returns.mean(), strategy_returns.std(ddof=1)
bh_mean, bh_std = raw_returns.mean(), raw_returns.std(ddof=1)
hits = np.sum(np.sign(tone_scores) == np.sign(raw_returns))

print("Strategy returns (%):", strategy_returns)
print(f"Strategy: mean={strat_mean:.4f}%, std={strat_std:.4f}%, Sharpe-like={strat_mean/strat_std:.4f}")
print(f"Buy-hold: mean={bh_mean:.4f}%, std={bh_std:.4f}%, Sharpe-like={bh_mean/bh_std:.4f}")
print(f"Hit rate: {hits}/{len(tone_scores)} = {hits/len(tone_scores):.4f}")

Strategy returns (%): [ 1.2  0.8  0.3 -0.4  2.1  1.5 -0.2  0.9]
Strategy: mean=0.7750%, std=0.8481%, Sharpe-like=0.9138
Buy-hold: mean=0.0750%, std=1.1829%, Sharpe-like=0.0634
Hit rate: 6/8 = 0.7500


## 4. Naive Bayes classification of 10-K risk-factor sentences

In [5]:
vocab2 = ["default", "borrower", "counterparty", "volatility", "rate", "currency"]

def count_words(sentence, vocab):
    tokens = sentence.lower().replace(",", "").replace(".", "").replace("'", "").split()
    return np.array([tokens.count(w) for w in vocab])

train_credit_sentences = [
    "A default by a major borrower or counterparty could trigger a chain of default across our loan portfolio.",
    "Our borrower base includes several highly leveraged parties, and a default by any significant borrower could impair our credit portfolio.",
    "We are exposed to counterparty risk if a significant counterparty were to default on its obligations under our agreements.",
]
train_market_sentences = [
    "Significant volatility in interest rate markets, combined with broader market volatility, could adversely affect the value of our trading positions.",
    "Changes in the benchmark rate or the market rate environment, combined with volatility in trading conditions, could affect our net interest income.",
    "Fluctuations in the currency exchange market and adverse currency movements, together with changes in the base lending rate, could reduce our reported earnings.",
]

train_credit = np.array([count_words(s, vocab2) for s in train_credit_sentences])
train_market = np.array([count_words(s, vocab2) for s in train_market_sentences])
print("Tokenized Credit counts:\n", train_credit)
print("Tokenized Market counts:\n", train_market)

alpha, V = 1.0, len(vocab2)
credit_counts, market_counts = train_credit.sum(axis=0), train_market.sum(axis=0)
credit_total, market_total = credit_counts.sum(), market_counts.sum()

p_word_credit = (credit_counts + alpha) / (credit_total + alpha * V)
p_word_market = (market_counts + alpha) / (market_total + alpha * V)

print("P(word|Credit):", p_word_credit.round(4))
print("P(word|Market):", p_word_market.round(4))

Tokenized Credit counts:
 [[2 1 1 0 0 0]
 [1 2 0 0 0 0]
 [1 0 2 0 0 0]]
Tokenized Market counts:
 [[0 0 0 2 1 0]
 [0 0 0 1 2 0]
 [0 0 0 0 1 2]]
P(word|Credit): [0.3125 0.25   0.25   0.0625 0.0625 0.0625]
P(word|Market): [0.0667 0.0667 0.0667 0.2667 0.3333 0.2   ]


In [6]:
prior_credit = prior_market = 0.5

# T3 is the ambiguous held-out sentence worked through by hand in the chapter text
t3_sentence = ("A default by a key counterparty during a period of heightened market volatility "
               "could be exacerbated by continued volatility in broader financial markets.")
t3_counts = count_words(t3_sentence, vocab2)
print("T3 tokenized counts:", t3_counts.tolist())

test_docs = {
    "T1": ([2, 1, 0, 0, 0, 0], "Credit"),
    "T2": ([0, 1, 2, 0, 0, 0], "Credit"),
    "T3": (t3_counts.tolist(), "Credit"),
    "T4": ([0, 0, 0, 2, 0, 1], "Market"),
    "T5": ([0, 0, 0, 0, 2, 1], "Market"),
    "T6": ([0, 0, 0, 1, 1, 0], "Market"),
}

def nb_scores(counts):
    counts = np.array(counts)
    sc = prior_credit * np.prod(p_word_credit ** counts)
    sm = prior_market * np.prod(p_word_market ** counts)
    return sc, sm

results = {}
for name, (counts, true_label) in test_docs.items():
    sc, sm = nb_scores(counts)
    pred = "Credit" if sc > sm else "Market"
    results[name] = (true_label, pred)
    print(f"{name}: true={true_label}, P(Credit|doc)={sc/(sc+sm):.4f}, predicted={pred}")

TP = sum(1 for v in results.values() if v[0]=="Credit" and v[1]=="Credit")
FN = sum(1 for v in results.values() if v[0]=="Credit" and v[1]=="Market")
FP = sum(1 for v in results.values() if v[0]=="Market" and v[1]=="Credit")
TN = sum(1 for v in results.values() if v[0]=="Market" and v[1]=="Market")
precision = TP/(TP+FP)
recall = TP/(TP+FN)
f1 = 2*precision*recall/(precision+recall)
print(f"\nConfusion matrix: TP={TP}, FN={FN}, FP={FP}, TN={TN}")
print(f"Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

T3 tokenized counts: [1, 0, 1, 2, 0, 0]
T1: true=Credit, P(Credit|doc)=0.9880, predicted=Credit
T2: true=Credit, P(Credit|doc)=0.9814, predicted=Credit
T3: true=Credit, P(Credit|doc)=0.4912, predicted=Market
T4: true=Market, P(Credit|doc)=0.0169, predicted=Market
T5: true=Market, P(Credit|doc)=0.0109, predicted=Market
T6: true=Market, P(Credit|doc)=0.0421, predicted=Market

Confusion matrix: TP=2, FN=1, FP=0, TN=3
Precision=1.0000, Recall=0.6667, F1=0.8000


## 5. Cosine similarity of toy word embeddings

In [7]:
embeddings = {
    "stock": np.array([0.90, 0.20]),
    "equity": np.array([0.85, 0.25]),
    "bond": np.array([0.10, 0.90]),
    "debt": np.array([0.15, 0.85]),
    "bullish": np.array([0.80, -0.30]),
    "bearish": np.array([-0.75, -0.35]),
}

def cosine(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

for a, b in [("stock","equity"), ("bond","debt"), ("stock","bond"), ("bullish","bearish"), ("stock","bullish")]:
    print(f"cos({a}, {b}) = {cosine(embeddings[a], embeddings[b]):.4f}")

cos(stock, equity) = 0.9977
cos(bond, debt) = 0.9980
cos(stock, bond) = 0.3234
cos(bullish, bearish) = -0.7000
cos(stock, bullish) = 0.8379


### 5b. Vector arithmetic: an analogy relation (Section 14.10)

Check whether stock - equity + bond reproduces debt exactly.

In [8]:
result = embeddings["stock"] - embeddings["equity"] + embeddings["bond"]
print(f"stock - equity + bond = {result}")
print(f"debt                   = {embeddings['debt']}")
print(f"Exact match? {np.allclose(result, embeddings['debt'])}")

# Exercise: bearish - bullish + stock
result_ex = embeddings["bearish"] - embeddings["bullish"] + embeddings["stock"]
print(f"\nExercise: bearish - bullish + stock = {result_ex.round(4)}")
for name, vec in embeddings.items():
    print(f"  cos(result, {name}) = {cosine(result_ex, vec):.4f}")

stock - equity + bond = [0.15 0.85]
debt                   = [0.15 0.85]
Exact match? True

Exercise: bearish - bullish + stock = [-0.65  0.15]
  cos(result, stock) = -0.9024
  cos(result, equity) = -0.8713
  cos(result, bond) = 0.1159
  cos(result, debt) = 0.0521
  cos(result, bullish) = -0.9913
  cos(result, bearish) = 0.7879


## 6. Scaled dot-product self-attention by hand

In [9]:
# Tokens: "stocks", "rallied", "today"
K = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
V_mat = np.array([[2, 0], [0, 2], [1, 1]], dtype=float)
Q1 = np.array([1, 0], dtype=float)  # query for token "stocks"

d_k = 2
scores = (K @ Q1) / np.sqrt(d_k)
weights = np.exp(scores) / np.exp(scores).sum()
output = weights @ V_mat

print("Raw scores:", scores.round(4))
print("Softmax weights:", weights.round(4))
print("Attention output for 'stocks':", output.round(4))

Raw scores: [0.7071 0.     0.7071]
Softmax weights: [0.4011 0.1978 0.4011]
Attention output for 'stocks': [1.2033 0.7967]


### 6b. Causal vs. bidirectional attention: encoder-only vs. decoder-only (Section 14.9)

Recompute "rallied"'s contextual representation, once with full bidirectional attention (encoder-only, e.g. BERT) and once with a causal mask hiding "today" (decoder-only, e.g. GPT).

In [10]:
tokens = ["stocks", "rallied", "today"]
Q_rallied = np.array([0, 1], dtype=float)

# Bidirectional (encoder-only): rallied attends to all three tokens
scores_bi = (K @ Q_rallied) / np.sqrt(d_k)
weights_bi = np.exp(scores_bi) / np.exp(scores_bi).sum()
output_bi = weights_bi @ V_mat
print("Bidirectional -- scores:", scores_bi.round(4), "weights:", weights_bi.round(4))
print("Bidirectional attention output for 'rallied':", output_bi.round(4))

# Causal (decoder-only): rallied (position 2) can only attend to stocks(1) and rallied(2)
K_causal = K[:2]
V_causal = V_mat[:2]
scores_causal = (K_causal @ Q_rallied) / np.sqrt(d_k)
weights_causal = np.exp(scores_causal) / np.exp(scores_causal).sum()
output_causal = weights_causal @ V_causal
print("\nCausal -- scores:", scores_causal.round(4), "weights:", weights_causal.round(4))
print("Causal attention output for 'rallied':", output_causal.round(4))

Bidirectional -- scores: [0.     0.7071 0.7071] weights: [0.1978 0.4011 0.4011]
Bidirectional attention output for 'rallied': [0.7967 1.2033]

Causal -- scores: [0.     0.7071] weights: [0.3302 0.6698]
Causal attention output for 'rallied': [0.6605 1.3395]


## 7. Gunning Fog readability index

In [11]:
def fog_index(total_words, sentences, complex_words):
    return 0.4 * ((total_words/sentences) + 100*(complex_words/total_words))

fog_a = fog_index(total_words=17, sentences=3, complex_words=0)
fog_b = fog_index(total_words=24, sentences=2, complex_words=14)
print(f"Paragraph A (plain): Fog index = {fog_a:.4f}")
print(f"Paragraph B (dense boilerplate): Fog index = {fog_b:.4f}")

# Paragraph C: a plain-English rewrite of Paragraph B, same substantive content
fog_c = fog_index(total_words=23, sentences=2, complex_words=3)
print(f"Paragraph C (plain-English rewrite of B): Fog index = {fog_c:.4f}")

# Exercise: a hypothetical rewrite with 20 words, 2 sentences, 1 complex word
fog_ex = fog_index(total_words=20, sentences=2, complex_words=1)
print(f"\nExercise (20 words, 1 complex word): Fog index = {fog_ex:.4f}")

Paragraph A (plain): Fog index = 2.2667
Paragraph B (dense boilerplate): Fog index = 28.1333
Paragraph C (plain-English rewrite of B): Fog index = 9.8174

Exercise (20 words, 1 complex word): Fog index = 6.0000


## 8. Retrieval-augmented generation (RAG): a toy retrieval step

In [12]:
kb_docs = {
    "Doc1": ("Company X reported Q3 revenue of $420 million, up 8% year over year.", np.array([0.90, 0.10, 0.00])),
    "Doc2": ("Company X's debt-to-equity ratio increased to 1.8 in Q3.", np.array([0.10, 0.90, 0.00])),
    "Doc3": ("Company Y announced a new product launch in Q4.", np.array([0.00, 0.00, 0.90])),
    "Doc4": ("Company X's Q3 net margin improved to 12.5% from 10.1%.", np.array([0.60, 0.50, 0.00])),
}
query_vec = np.array([0.95, 0.05, 0.00])

sims = {name: cosine(query_vec, vec) for name, (text, vec) in kb_docs.items()}
for name, (text, vec) in kb_docs.items():
    print(f"cos(query, {name}) = {sims[name]:.4f}   [{text}]")

ranked = sorted(sims.items(), key=lambda kv: kv[1], reverse=True)
print("\nRanked retrieval order:", [name for name, _ in ranked])
print("Top-1 retrieved document:", ranked[0][0], "-", kb_docs[ranked[0][0]][0])

cos(query, Doc1) = 0.9983   [Company X reported Q3 revenue of $420 million, up 8% year over year.]
cos(query, Doc2) = 0.1625   [Company X's debt-to-equity ratio increased to 1.8 in Q3.]
cos(query, Doc3) = 0.0000   [Company Y announced a new product launch in Q4.]
cos(query, Doc4) = 0.8008   [Company X's Q3 net margin improved to 12.5% from 10.1%.]

Ranked retrieval order: ['Doc1', 'Doc4', 'Doc2', 'Doc3']
Top-1 retrieved document: Doc1 - Company X reported Q3 revenue of $420 million, up 8% year over year.


### 8c. Evaluating RAG: precision@k and a faithfulness score (Section 14.15)

Precision@k measures the retrieval step (only Doc1 truly answers the revenue question); the faithfulness score measures the generation step (does the generated answer's embedding match the retrieved source's?).

In [13]:
retrieved_order = [name for name, _ in ranked]
relevant = {"Doc1": True, "Doc4": False, "Doc2": False, "Doc3": False}

for k in [1, 2, 3]:
    topk = retrieved_order[:k]
    prec = sum(relevant[d] for d in topk) / k
    print(f"Precision@{k}: {prec:.4f}")

Doc1_vec = kb_docs["Doc1"][1]
grounded_answer = np.array([0.89, 0.12, 0.00])
hallucinated_answer = np.array([0.30, 0.85, 0.00])

print(f"\nFaithfulness (grounded answer): {cosine(grounded_answer, Doc1_vec):.4f}")
print(f"Faithfulness (hallucinated answer): {cosine(hallucinated_answer, Doc1_vec):.4f}")

# Exercise: a fifth document, Doc5, also not relevant, retrieved fourth
sims_ex = dict(sims)
Doc5_vec = np.array([0.75, 0.30, 0.00])
sims_ex["Doc5"] = cosine(query_vec, Doc5_vec)
ranked_ex = sorted(sims_ex.items(), key=lambda kv: kv[1], reverse=True)
print("\nExercise -- ranked order with Doc5:", [name for name, _ in ranked_ex])
relevant_ex = {**relevant, "Doc5": False}
prec4 = sum(relevant_ex[d] for d, _ in ranked_ex[:4]) / 4
print(f"Exercise -- Precision@4: {prec4:.4f}")

# Exercise: faithfulness of a third candidate answer
answer_ex = np.array([0.80, 0.25, 0.00])
print(f"Exercise -- Faithfulness of (0.80, 0.25, 0.00): {cosine(answer_ex, Doc1_vec):.4f}")

Precision@1: 1.0000
Precision@2: 0.5000
Precision@3: 0.3333

Faithfulness (grounded answer): 0.9997
Faithfulness (hallucinated answer): 0.4349

Exercise -- ranked order with Doc5: ['Doc1', 'Doc5', 'Doc4', 'Doc2', 'Doc3']
Exercise -- Precision@4: 0.2500
Exercise -- Faithfulness of (0.80, 0.25, 0.00): 0.9816


## 8b. A GAN's adversarial game, evaluated at two training stages

A toy discriminator and generator for synthesizing fraud-amount z-scores (extending Chapter 13's SMOTE discussion with entirely synthetic, rather than interpolated, examples), evaluated at an early stage (an untrained generator producing an unrealistic sample) and a later stage (a generator that has learned to produce a more realistic sample), showing $D(G(z))$ rise toward $D(x_{\text{real}})$ as training proceeds, exactly as the minimax equation above predicts.

In [14]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Toy discriminator D(x) = sigmoid(w_d*x + b_d), fixed illustrative weights
# representing a partially trained discriminator at some point in training
w_d, b_d = 2.0, -5.0

# Real fraud amount z-score, reusing Chapter 13's transaction 1 (z=3.1)
x_real = 3.1
D_real = sigmoid(w_d * x_real + b_d)
print(f"D(x_real={x_real}) = {D_real:.4f}")

# Early-training generator: produces an unrealistic sample, close to a typical
# LEGITIMATE transaction's amount z-score rather than a fraudulent one
G_early = 1.0
D_early = sigmoid(w_d * G_early + b_d)
print(f"Early generator: G(z)={G_early}, D(G(z))={D_early:.4f}")

# Later-training generator: has learned to produce a more realistic (higher, fraud-like) sample
G_later = 2.9
D_later = sigmoid(w_d * G_later + b_d)
print(f"Later generator: G(z)={G_later}, D(G(z))={D_later:.4f}")

# Discriminator objective (wants to MAXIMIZE): log D(x_real) + log(1 - D(G(z)))
disc_obj_early = np.log(D_real) + np.log(1 - D_early)
disc_obj_later = np.log(D_real) + np.log(1 - D_later)
print(f"\nDiscriminator objective, early stage: {disc_obj_early:.4f}")
print(f"Discriminator objective, later stage: {disc_obj_later:.4f}")

# Generator's non-saturating loss -log(D(G(z))), wants to MINIMIZE
gen_loss_early = -np.log(D_early)
gen_loss_later = -np.log(D_later)
print(f"\nGenerator loss, early stage: {gen_loss_early:.4f}")
print(f"Generator loss, later stage: {gen_loss_later:.4f}")

print(f"\nD_real - D(G(z)) gap, early stage: {D_real - D_early:.4f}")
print(f"D_real - D(G(z)) gap, later stage: {D_real - D_later:.4f}")

D(x_real=3.1) = 0.7685
Early generator: G(z)=1.0, D(G(z))=0.0474
Later generator: G(z)=2.9, D(G(z))=0.6900

Discriminator objective, early stage: -0.3119
Discriminator objective, later stage: -1.4344

Generator loss, early stage: 3.0486
Generator loss, later stage: 0.3711

D_real - D(G(z)) gap, early stage: 0.7211
D_real - D(G(z)) gap, later stage: 0.0786


## 9. One forward diffusion step on a toy stress-scenario variable

In [15]:
x0 = 1.00      # baseline scenario multiplier (1.0 = no stress)
alpha_t = 0.80 # signal-retention coefficient at step t
epsilon = 0.50 # standard-normal noise draw

xt = np.sqrt(alpha_t) * x0 + np.sqrt(1 - alpha_t) * epsilon
print(f"x0={x0}, alpha_t={alpha_t}, epsilon={epsilon}")
print(f"x_t = sqrt(alpha_t)*x0 + sqrt(1-alpha_t)*epsilon = {xt:.4f}")

alpha_t2 = 0.55
xt2 = np.sqrt(alpha_t2) * x0 + np.sqrt(1 - alpha_t2) * epsilon
print(f"With alpha_t={alpha_t2} (later/noisier step): x_t = {xt2:.4f}")

x0=1.0, alpha_t=0.8, epsilon=0.5
x_t = sqrt(alpha_t)*x0 + sqrt(1-alpha_t)*epsilon = 1.1180
With alpha_t=0.55 (later/noisier step): x_t = 1.0770


## 10. RLHF reward-model preference probability (Bradley-Terry)

In [16]:
r_A = 2.1  # reward score for a well-grounded, accurate summary
r_B = 0.8  # reward score for a fluent but subtly inaccurate summary

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

p_A_preferred = sigmoid(r_A - r_B)
print(f"r_A={r_A}, r_B={r_B}, r_A - r_B={r_A-r_B:.4f}")
print(f"P(A preferred over B) = sigmoid(r_A - r_B) = {p_A_preferred:.4f}")

r_A=2.1, r_B=0.8, r_A - r_B=1.3000
P(A preferred over B) = sigmoid(r_A - r_B) = 0.7858


### 10b. Parameter-efficient fine-tuning: LoRA (Section 14.16)

Low-rank adaptation replaces a full d x d weight update with two low-rank matrices of size d x r and r x d.

In [17]:
d, r = 4096, 8
full_params = d * d
lora_params = 2 * d * r
print(f"Full fine-tuning parameters: {full_params:,}")
print(f"LoRA (rank {r}) parameters: {lora_params:,}")
print(f"Reduction factor: {full_params / lora_params:.1f}x")

# Exercise: rank increased to 32
r_ex = 32
lora_params_ex = 2 * d * r_ex
print(f"\nExercise (rank {r_ex}): LoRA parameters: {lora_params_ex:,}, "
      f"reduction factor: {full_params / lora_params_ex:.1f}x")

Full fine-tuning parameters: 16,777,216
LoRA (rank 8) parameters: 65,536
Reduction factor: 256.0x

Exercise (rank 32): LoRA parameters: 262,144, reduction factor: 64.0x


## 11. A two-step LLM agent computing a P/E ratio

In [18]:
eps_q3 = 3.15
price_q3 = 57.87
pe_ratio = price_q3 / eps_q3
print(f"Retrieved EPS (Q3) = ${eps_q3}, Retrieved price (Q3 close) = ${price_q3}")
print(f"P/E = price / EPS = {pe_ratio:.4f}")

# Step 3: cross-check against an independent, vendor-reported P/E
vendor_pe = 19.10
discrepancy = (vendor_pe - pe_ratio) / pe_ratio
tolerance = 0.02
print(f"\nVendor-reported P/E = {vendor_pe}")
print(f"Discrepancy = {discrepancy:.4%}")
print(f"Flagged for review? {abs(discrepancy) > tolerance}")

# Exercise: vendor P/E of 18.50 instead of 19.10
vendor_pe_ex = 18.50
discrepancy_ex = (vendor_pe_ex - pe_ratio) / pe_ratio
print(f"\nExercise -- vendor P/E={vendor_pe_ex}: discrepancy={discrepancy_ex:.4%}, "
      f"flagged? {abs(discrepancy_ex) > tolerance}")

Retrieved EPS (Q3) = $3.15, Retrieved price (Q3 close) = $57.87
P/E = price / EPS = 18.3714

Vendor-reported P/E = 19.1
Discrepancy = 3.9658%
Flagged for review? True

Exercise -- vendor P/E=18.5: discrepancy=0.6998%, flagged? False


## 12. Mixture-of-experts sparse activation fraction

In [19]:
n_experts = 8
params_per_expert_b = 6  # billions
top_k = 2
total_expert_params = n_experts * params_per_expert_b
active_expert_params = top_k * params_per_expert_b
active_fraction = active_expert_params / total_expert_params
print(f"Total expert parameters: {total_expert_params}B, active per token: {active_expert_params}B")
print(f"Active fraction = {active_fraction:.4f} = {active_fraction*100:.1f}%")

Total expert parameters: 48B, active per token: 12B
Active fraction = 0.2500 = 25.0%


## 13. Quantization: memory footprint of a 50B-parameter model

In [20]:
params = 50e9
bytes_fp16 = params * 2
bytes_int8 = params * 1
bytes_int4 = params * 0.5

for name, b in [("FP16", bytes_fp16), ("INT8", bytes_int8), ("INT4", bytes_int4)]:
    print(f"{name}: {b/1e9:.1f} GB")

print(f"\nReduction FP16 -> INT8: {bytes_fp16/bytes_int8:.1f}x")
print(f"Reduction FP16 -> INT4: {bytes_fp16/bytes_int4:.1f}x")

# Exercise: 175B-parameter model
params_ex = 175e9
bytes_fp16_ex = params_ex * 2
bytes_int8_ex = params_ex * 1
bytes_int4_ex = params_ex * 0.5
print(f"\nExercise (175B params) -- FP16: {bytes_fp16_ex/1e9:.1f} GB, "
      f"INT8: {bytes_int8_ex/1e9:.1f} GB, INT4: {bytes_int4_ex/1e9:.1f} GB, "
      f"reduction FP16->INT4: {bytes_fp16_ex/bytes_int4_ex:.1f}x")

FP16: 100.0 GB
INT8: 50.0 GB
INT4: 25.0 GB

Reduction FP16 -> INT8: 2.0x
Reduction FP16 -> INT4: 4.0x

Exercise (175B params) -- FP16: 350.0 GB, INT8: 175.0 GB, INT4: 87.5 GB, reduction FP16->INT4: 4.0x


### 13b. API cost versus self-hosting at scale (Section 14.16)

Quantization's memory savings translate into a real dollar comparison once query volume is large enough.

In [21]:
queries_per_day = 40_000
tokens_per_query = 1000
api_price_per_1k = 0.03

daily_api_cost = queries_per_day * (tokens_per_query / 1000) * api_price_per_1k
monthly_api_cost = daily_api_cost * 30
print(f"Daily API cost: ${daily_api_cost:,.0f}, monthly: ${monthly_api_cost:,.0f}")

throughput_per_hour = 4000
gpu_cost_per_hour = 20
hours_needed = queries_per_day / throughput_per_hour
daily_selfhost_cost = hours_needed * gpu_cost_per_hour
monthly_selfhost_cost = daily_selfhost_cost * 30
print(f"Hours needed/day: {hours_needed:.1f}, daily self-hosted cost: ${daily_selfhost_cost:,.0f}, "
      f"monthly: ${monthly_selfhost_cost:,.0f}")
print(f"Ratio (API / self-hosted): {monthly_api_cost / monthly_selfhost_cost:.1f}x")

# Exercise: volume doubles to 80,000 queries/day
queries_per_day_ex = 80_000
monthly_api_cost_ex = queries_per_day_ex * (tokens_per_query / 1000) * api_price_per_1k * 30
hours_needed_ex = queries_per_day_ex / throughput_per_hour
monthly_selfhost_cost_ex = hours_needed_ex * gpu_cost_per_hour * 30
print(f"\nExercise (80,000 queries/day) -- monthly API: ${monthly_api_cost_ex:,.0f}, "
      f"monthly self-hosted: ${monthly_selfhost_cost_ex:,.0f}, "
      f"ratio: {monthly_api_cost_ex / monthly_selfhost_cost_ex:.1f}x")

Daily API cost: $1,200, monthly: $36,000
Hours needed/day: 10.0, daily self-hosted cost: $200, monthly: $6,000
Ratio (API / self-hosted): 6.0x

Exercise (80,000 queries/day) -- monthly API: $72,000, monthly self-hosted: $12,000, ratio: 6.0x
